# Resultados del pronóstico municipal 6→6, clúster 1

[← Metodología común](metodologia-pronostico-6x6.ipynb)

Este notebook presenta exclusivamente la configuración y los resultados de
`cluster1`. El entrenamiento comienza en `2016-01-01` y el detalle inicial se
centra en **Providencia**.


In [9]:
# Configuración editable del notebook
from utils.forecasting_cache import ForecastCacheConfig

CLUSTER_COLUMN = "cluster1"
TRAINING_START = "2016-01-01"
PREFERRED_MUNICIPALITY = "Providencia"

MODEL_CACHE = ForecastCacheConfig(
    directory="model_cache",
    enabled=True,
    refresh=False,
)


## Composición del clúster 1

La celda resuelve las comunas marcadas en `cluster1` y deja visible la configuración
efectiva antes de cargar los datos.


In [10]:
import pandas as pd
from IPython.display import HTML, display

from utils.data import data_loader
from utils.forecasting_cache import summarize_cache_report
from utils.forecasting_workflow import (
    build_municipality_forecast_view,
    macro_micro_gap,
    resolve_cluster_municipalities,
    run_cluster_forecasting_workflow,
)
from utils.municipal_income import build_municipalities_income_history
from utils.notebook_display import collapsible_stdout, display_collapsible
from utils.plots import (
    plot_annual_income_share_lines,
    plot_municipality_forecast_evaluation,
)

clusters = pd.read_csv("clusters.csv")
municipalities = resolve_cluster_municipalities(clusters, CLUSTER_COLUMN)
display_collapsible(
    "Ver configuración efectiva",
    pd.DataFrame(
        {
            "cluster": [CLUSTER_COLUMN],
            "comunas": [len(municipalities)],
            "inicio de entrenamiento": [pd.Timestamp(TRAINING_START)],
            "comuna para detalle": [PREFERRED_MUNICIPALITY],
            "caché habilitado": [MODEL_CACHE.enabled],
            "refrescar caché": [MODEL_CACHE.refresh],
            "directorio caché": [str(MODEL_CACHE.directory)],
        }
    )
)
municipality_label = "comuna" if len(municipalities) == 1 else "comunas"
municipality_items = "".join(
    f"<li>{municipality}</li>" for municipality in municipalities
)
display_collapsible(
    f"Ver {len(municipalities)} {municipality_label} del clúster",
    HTML(
        f'<ul class="cluster-members__list">{municipality_items}</ul>'
    )
)


,cluster,comunas,inicio de entrenamiento,comuna para detalle,caché habilitado,refrescar caché,directorio caché
0,cluster1,5,2016-01-01,Providencia,True,False,model_cache


## Historia descriptiva del clúster 1

Se muestran trayectorias independientes para las comunas de `cluster1` antes de
presentar la evaluación predictiva.


In [11]:
with collapsible_stdout("Ver registro de carga de datos"):
    presupuesto_cluster = data_loader(municipalities=municipalities)
historial_cluster = build_municipalities_income_history(
    presupuesto_cluster,
    municipalities=municipalities,
)

for municipality in municipalities:
    municipality_history = historial_cluster.loc[
        historial_cluster["Nombre Municipio"].eq(municipality)
    ]
    figure = plot_annual_income_share_lines(municipality_history)
    year_count = int(municipality_history["Ejercicio"].nunique())
    figure.update_layout(width=max(900, year_count * 90 + 240), autosize=False)
    figure_html = figure.to_html(
        full_html=False,
        include_plotlyjs="cdn",
        config={"responsive": False, "displaylogo": False},
    )
    display(
        HTML(
            '<div style="max-width:100%; overflow-x:auto; padding-bottom:1rem;">'
            f"{figure_html}</div>"
        )
    )


## Ejecución y trazabilidad del clúster 1

El bloque ejecuta el workflow conjunto para `cluster1`. El contrato temporal, las
métricas y el comportamiento del caché están documentados en la
[metodología común](metodologia-pronostico-6x6.ipynb).


In [12]:
with collapsible_stdout("Ver registro de entrenamiento conjunto"):
    workflow = run_cluster_forecasting_workflow(
        presupuesto_cluster,
        municipalities,
        cluster_label=CLUSTER_COLUMN,
        training_start=TRAINING_START,
        progress=True,
        cache_config=MODEL_CACHE,
    )

display_collapsible("Ver configuración del workflow", workflow.configuration)
display_collapsible("Ver ventanas de entrenamiento", workflow.training_windows)
display_collapsible("Ver progresión de modelos", workflow.model_progression)
display_collapsible(
    "Ver resumen del caché",
    summarize_cache_report(workflow.cache_report),
)

cache_by_stage = (
    workflow.cache_report.groupby(
        ["stage", "artifact_type", "status"], sort=False, dropna=False
    )
    .agg(artefactos=("key", "size"), gb=("size_bytes", lambda x: x.sum() / 1024**3))
    .reset_index()
)
display_collapsible("Ver artefactos de caché por etapa", cache_by_stage)


,cluster,comunas,inicio_entrenamiento_solicitado,primer_corte_tuning,ultimo_corte_tuning,primer_corte_validacion,ultimo_corte_validacion,fin_entrenamiento_final,entrada_final,test_congelado,ventanas_entrenamiento_final,variables_mensuales,objetivos_directos
0,cluster1,5,2016-01-01,2018-06-01,2022-12-01,2023-06-01,2024-06-01,2025-06-01,julio-diciembre 2025,enero-junio 2026,515,30,30


,Nombre Municipio,primera_entrada,ultimo_objetivo,ventanas
0,Alhué,2016-01-01,2025-06-01,103
1,Huechuraba,2016-01-01,2025-06-01,103
2,Lo Barnechea,2016-01-01,2025-06-01,103
3,Providencia,2016-01-01,2025-06-01,103
4,Vitacura,2016-01-01,2025-06-01,103


,nivel,modelo,complejidad
0,1,Persistencia,Repite el último vector mensual; no se entrena.
1,2,Ridge global,Relación lineal regularizada; comuna y mes one...
2,3,ExtraTrees global,Ensamble no lineal; comuna y mes one-hot.
3,4,CatBoost global,Boosting no lineal; comuna y mes categóricos n...


,hits,misses,modelos_cargados,evaluaciones_cargadas,reutilizaciones_memoria,artefactos_escritos,artefactos_corruptos_aislados,gb_cargados,gb_escritos,gb_artefactos_utilizados
0,117,0,3,114,0,0,0,0.051829,0.0,0.051829


,stage,artifact_type,status,artefactos,gb
0,joint_tuning,evaluation,hit,110,0.000418
1,joint_validation,evaluation,hit,3,0.000021
2,joint_final_model,model,hit,3,0.051384
3,joint_test,evaluation,hit,1,0.000007


## Hiperparámetros elegidos para el clúster 1

La tabla identifica las configuraciones retenidas para `cluster1` antes de comparar
familias en validación.


In [13]:
tuning_table = workflow.tuning_results.copy()
tuning_table["WAPE macro (%)"] = (100 * tuning_table["wape_macro"]).round(2)
tuning_table["WAPE micro (%)"] = (100 * tuning_table["wape_micro"]).round(2)
tuning_table["MAE (MM CLP)"] = tuning_table["mae_mm_clp"].round(2)
selected_parameters = pd.DataFrame(
    [
        {
            "modelo": spec.name,
            "candidate_id": spec.candidate_id,
            "parametros": dict(spec.params),
        }
        for spec in workflow.selected_specs
    ]
)
display_collapsible(
    "Ver resultados de tuning",
    tuning_table[
        [
            "familia",
            "modelo_candidato",
            "parametros",
            "seleccionado",
            "WAPE macro (%)",
            "WAPE micro (%)",
            "MAE (MM CLP)",
        ]
    ]
)
display_collapsible("Ver hiperparámetros seleccionados", selected_parameters)


,familia,modelo_candidato,parametros,seleccionado,WAPE macro (%),WAPE micro (%),MAE (MM CLP)
0,ridge,Ridge global [alpha=1],{'alpha': 1.0},True,31.61,22.03,1125.44
1,ridge,Ridge global [alpha=0.1],{'alpha': 0.1},False,32.63,22.31,1139.91
2,ridge,Ridge global [alpha=10],{'alpha': 10.0},False,33.57,25.66,1310.91
3,extra_trees,"ExtraTrees global [depth=none, leaf=1]","{'n_estimators': 400, 'max_depth': None, 'min_...",True,28.10,23.10,1180.30
4,extra_trees,"ExtraTrees global [depth=none, leaf=3]","{'n_estimators': 400, 'max_depth': None, 'min_...",False,29.55,24.00,1226.22
5,extra_trees,"ExtraTrees global [depth=12, leaf=3]","{'n_estimators': 400, 'max_depth': 12, 'min_sa...",False,30.02,24.62,1257.75
6,extra_trees,"ExtraTrees global [depth=12, leaf=1]","{'n_estimators': 400, 'max_depth': 12, 'min_sa...",False,30.10,25.70,1313.01
7,catboost,"CatBoost global [depth=6, iterations=500]","{'iterations': 500, 'depth': 6, 'learning_rate...",True,30.79,25.95,1325.71
8,catboost,"CatBoost global [depth=4, iterations=500]","{'iterations': 500, 'depth': 4, 'learning_rate...",False,31.92,26.97,1377.61
9,catboost,"CatBoost global [depth=6, iterations=300]","{'iterations': 300, 'depth': 6, 'learning_rate...",False,32.12,27.47,1403.42


,modelo,candidate_id,parametros
0,Ridge global,ridge_alpha_1,{'alpha': 1.0}
1,ExtraTrees global,extra_trees_depth_none_leaf_1,"{'n_estimators': 400, 'max_depth': None, 'min_..."
2,CatBoost global,catboost_depth_6_iterations_500,"{'iterations': 500, 'depth': 6, 'learning_rate..."


## Selección histórica del clúster 1

Este bloque muestra el ganador congelado de `cluster1` y sus métricas por grupo de
ingreso antes de abrir el test de 2026.


In [14]:
validation_ranking = workflow.validation_ranking.copy()
validation_ranking["WAPE macro (%)"] = (
    100 * validation_ranking["wape_macro"]
).round(2)
validation_ranking["WAPE micro (%)"] = (
    100 * validation_ranking["wape_micro"]
).round(2)

validation_metrics = workflow.validation_summary.copy()
validation_metrics["WAPE macro (%)"] = (
    100 * validation_metrics["wape_macro"]
).round(2)
validation_metrics["WAPE micro (%)"] = (
    100 * validation_metrics["wape_micro"]
).round(2)
validation_metrics["MAE (MM CLP)"] = validation_metrics["mae_mm_clp"].round(2)
validation_metrics["Sesgo (MM CLP)"] = (
    validation_metrics["sesgo_micro_mm_clp"].round(2)
)

display_collapsible(
    "Ver ranking de validación",
    validation_ranking[
        [
            "ranking",
            "seleccion_validacion",
            "modelo",
            "WAPE macro (%)",
            "WAPE micro (%)",
            "mae_mm_clp",
            "sesgo_micro_mm_clp",
        ]
    ]
)
display_collapsible(
    "Ver métricas de validación por ingreso",
    validation_metrics[
        [
            "modelo",
            "grupo_ingreso",
            "WAPE macro (%)",
            "WAPE micro (%)",
            "MAE (MM CLP)",
            "Sesgo (MM CLP)",
        ]
    ]
)
display_collapsible(
    "Ver modelo seleccionado por validación",
    workflow.selected_model_name,
)


,ranking,seleccion_validacion,modelo,WAPE macro (%),WAPE micro (%),mae_mm_clp,sesgo_micro_mm_clp
0,1,True,ExtraTrees global,22.48,20.71,1429.203237,-1131.222871
1,2,False,Ridge global,30.13,25.26,1742.770736,340.759796
2,3,False,CatBoost global,30.35,26.91,1856.700167,-1539.450122
3,4,False,Persistencia,56.03,54.77,3778.843687,-1154.029019


,modelo,grupo_ingreso,WAPE macro (%),WAPE micro (%),MAE (MM CLP),Sesgo (MM CLP)
0,CatBoost global,FCM,29.98,29.63,74.06,-45.04
1,CatBoost global,IPP,31.17,27.11,1620.21,-1297.10
2,CatBoost global,Otros ingresos,63.01,62.25,193.34,-132.26
3,CatBoost global,Total disponible,30.35,26.91,1856.70,-1539.45
4,CatBoost global,Transferencias corrientes,80.04,41.68,141.27,-92.90
5,CatBoost global,Transferencias de capital,425.71,248.31,67.26,27.84
6,ExtraTrees global,FCM,27.14,26.03,65.06,-32.17
7,ExtraTrees global,IPP,22.81,20.73,1238.99,-992.27
8,ExtraTrees global,Otros ingresos,60.43,56.82,176.47,-71.82
9,ExtraTrees global,Total disponible,22.48,20.71,1429.20,-1131.22


## Test congelado de 2026 para el clúster 1

La comparación fuera de muestra de `cluster1` aplica la cobertura observada definida
en la [metodología común](metodologia-pronostico-6x6.ipynb).


In [15]:
test_ranking = workflow.test_ranking.copy()
test_ranking["WAPE macro (%)"] = (100 * test_ranking["wape_macro"]).round(2)
test_ranking["WAPE micro (%)"] = (100 * test_ranking["wape_micro"]).round(2)

test_metrics = workflow.test_summary.copy()
test_metrics["WAPE macro (%)"] = (100 * test_metrics["wape_macro"]).round(2)
test_metrics["WAPE micro (%)"] = (100 * test_metrics["wape_micro"]).round(2)
test_metrics["MAE (MM CLP)"] = test_metrics["mae_mm_clp"].round(2)
test_metrics["Sesgo (MM CLP)"] = test_metrics["sesgo_micro_mm_clp"].round(2)

display_collapsible(
    "Ver ranking del test 2026",
    test_ranking[
        [
            "ranking",
            "mejor_test",
            "modelo",
            "WAPE macro (%)",
            "WAPE micro (%)",
            "mae_mm_clp",
            "sesgo_micro_mm_clp",
        ]
    ]
)
display_collapsible(
    "Ver métricas del test 2026 por ingreso",
    test_metrics[
        [
            "modelo",
            "grupo_ingreso",
            "WAPE macro (%)",
            "WAPE micro (%)",
            "MAE (MM CLP)",
            "Sesgo (MM CLP)",
            "meses_evaluados",
        ]
    ]
)

if len(workflow.municipalities) == 1:
    assert macro_micro_gap(workflow.test_summary).fillna(0).le(1e-12).all()


,ranking,mejor_test,modelo,WAPE macro (%),WAPE micro (%),mae_mm_clp,sesgo_micro_mm_clp
0,1,True,ExtraTrees global,19.69,19.52,1561.959031,-1018.773827
1,2,False,CatBoost global,27.19,23.48,1878.906014,-1199.898419
2,3,False,Ridge global,37.36,35.07,2805.978503,1018.290687
3,4,False,Persistencia,58.43,54.97,4398.543257,-609.908959


,modelo,grupo_ingreso,WAPE macro (%),WAPE micro (%),MAE (MM CLP),Sesgo (MM CLP),meses_evaluados
0,CatBoost global,FCM,26.14,25.06,57.91,13.58,27.0
1,CatBoost global,IPP,28.92,23.57,1572.74,-784.43,27.0
2,CatBoost global,Otros ingresos,57.68,62.16,400.89,-360.60,27.0
3,CatBoost global,Total disponible,27.19,23.48,1878.91,-1199.90,27.0
4,CatBoost global,Transferencias corrientes,61.44,33.23,144.56,-90.20,27.0
5,CatBoost global,Transferencias de capital,1397.73,286.34,54.42,21.75,27.0
6,ExtraTrees global,FCM,19.52,16.96,39.20,-7.40,27.0
7,ExtraTrees global,IPP,19.98,18.85,1257.38,-688.16,27.0
8,ExtraTrees global,Otros ingresos,53.35,58.26,375.73,-295.80,27.0
9,ExtraTrees global,Total disponible,19.69,19.52,1561.96,-1018.77,27.0


## Detalle inicial de Providencia

`PREFERRED_MUNICIPALITY` permite cambiar la comuna inspeccionada dentro de `cluster1`
sin alterar la selección ni el ranking ya calculados.


In [16]:
municipality_view = build_municipality_forecast_view(
    workflow,
    PREFERRED_MUNICIPALITY,
)
display_collapsible(
    f"Ver métricas de {municipality_view.municipality}",
    municipality_view.metrics,
)

forecast_figure = plot_municipality_forecast_evaluation(
    municipality_view.actual,
    municipality_view.forecasts,
    municipality_view.metrics,
    municipality=municipality_view.municipality,
)
display(
    HTML(
        forecast_figure.to_html(
            full_html=False,
            include_plotlyjs="cdn",
            config={"responsive": True, "displaylogo": False},
        )
    )
)


,Nombre Municipio,modelo,grupo_ingreso,mae_mm_clp,sesgo_mm_clp,suma_error_absoluto,suma_observado_absoluto,wape,meses_evaluados
0,Providencia,CatBoost global,FCM,81.357521,48.077127,488.145124,1583.045087,0.308358,6.0
1,Providencia,CatBoost global,IPP,1642.263182,-698.735863,9853.579092,52310.198732,0.188368,6.0
2,Providencia,CatBoost global,Otros ingresos,206.001294,-206.001294,1236.007764,3414.135390,0.362027,6.0
3,Providencia,CatBoost global,Total disponible,1768.065779,-1202.416507,10608.394673,67394.750717,0.157407,6.0
4,Providencia,CatBoost global,Transferencias corrientes,433.856647,-433.856647,2603.139879,9974.154784,0.260989,6.0
5,Providencia,CatBoost global,Transferencias de capital,88.100170,88.100170,528.601021,113.216724,4.668931,6.0
6,Providencia,ExtraTrees global,FCM,42.320018,10.393364,253.920106,1583.045087,0.160400,6.0
7,Providencia,ExtraTrees global,IPP,2037.911664,-726.535421,12227.469983,52310.198732,0.233749,6.0
8,Providencia,ExtraTrees global,Otros ingresos,151.654978,-151.654978,909.929871,3414.135390,0.266518,6.0
9,Providencia,ExtraTrees global,Total disponible,2097.772546,-1037.547362,12586.635277,67394.750717,0.186760,6.0
